In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)
import calendar
from datetime import datetime, timedelta

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px
import plotly.graph_objects as go
zema = ZemaManager()


In [0]:
start_date = datetime(2018, 1, 1)
end_date = datetime(2030, 1, 1)

corn_factor_bu_to_mt=0.3937

# EURUSD

In [0]:


hrs= zema.get_curve(curve='P-FUTURE-CBOT-INPUT-WHEAT-HRS-USDc-BU', period=f"{start_date}::{end_date}")
hrs=hrs[hrs['observation']=='Last']
hrs

# Wheat
## SPOT KW EVOLUTION 

In [0]:
hrw=zema.get_curve(curve="P-FUTURE-CBOT-INPUT-HRW Wheat-USDc-BU", period=f"{start_date}::{end_date}")
hrw=hrw[hrw['observation']=='Settle']
hrw=hrw[['date','value','contract_year','contract_month']]
hrw['day']=hrw['date'].dt.day
hrw['month']=hrw['date'].dt.month
# CBOT month codes (if you want codes like Z25, H26, ...)
month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

def get_spot_contract(row):
    d = row['date']
    y = d.year
    m = d.month
    day = d.day

    # Sep 20 -> Dec 12  => Dec of current year
    if (m == 9 and day >= 13) or (m in [10, 11]) or (m == 12 and day <= 12):
        return (y, 12)

    # Dec 20 -> Dec 31 => March of NEXT year (roll happens at year end)
    if m == 12 and day >= 13:
        return (y + 1, 3)

    # Jan 1 -> Mar 12  => March of CURRENT year
    if (m in [1, 2]) or (m == 3 and day <= 12):
        return (y, 3)

    # Mar 20 -> May 12 => May of current year
    if (m == 3 and day >= 13) or (m == 4) or (m == 5 and day <= 12):
        return (y, 5)

    # May 13 -> Jul 12 => July of current year
    if (m == 5 and day >= 13) or (m == 6) or (m == 7 and day <= 12):
        return (y, 7)

    # Jul 13 -> Sep 12 => September of current year
    if (m == 7 and day >= 13) or (m == 8) or (m == 9 and day <= 12):
        return (y, 9)

    # safety fallback (shouldn't be hit with your rules)
    return (y, m)


# Apply to your hrw dataframe
hrw['spot_year'], hrw['spot_month'] = zip(*hrw.apply(get_spot_contract, axis=1))
hrw['spot_code'] = hrw.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the contract rows that correspond to the rolling spot
spot_hrw = hrw[
    (hrw['contract_year'] == hrw['spot_year']) &
    (hrw['contract_month'] == hrw['spot_month'])
].copy()

spot_hrw = spot_hrw.sort_values('date')



In [0]:
kc_hrw_spot = go.Figure()

# Add the rolling spot contract line
kc_hrw_spot.add_trace(go.Scatter(
    x=spot_hrw['date'],
    y=spot_hrw['value'],
    mode='lines',
    name="HRW Spot",
    line=dict(color="blue", width=2)
))

# Add markers when the contract changes
roll_dates = spot_hrw.loc[spot_hrw['spot_code'].shift() != spot_hrw['spot_code'], 'date']
roll_labels = spot_hrw.loc[spot_hrw['spot_code'].shift() != spot_hrw['spot_code'], 'spot_code']

for d, code in zip(roll_dates, roll_labels):
    kc_hrw_spot.add_vline(x=d, line_dash="dash", line_color="gray")
    kc_hrw_spot.add_annotation(
        x=d, y=spot_hrw['value'].max(),
        text=f"{code}",
        showarrow=False,
        yshift=20,
        font=dict(size=10, color="gray")
    )

# Layout settings
kc_hrw_spot.update_layout(
    title="KC HRW Rolling Spot Contract",
    xaxis_title="Date",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white"
)

kc_hrw_spot.show()
kc_hrw_spot_html = kc_hrw_spot.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
df_prices=hrw[['date', 'value', 'contract_year', 'contract_month', 'day', 'month',
       'spot_year', 'spot_month']]

month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# ---------------------------
# Assign virtual date for seasonal alignment (like spreads)
# ---------------------------
def assign_virtual_date_with_history(row):
    d = row['date']
    cm = row['contract_month']
    year = row['contract_year']  # the contract year
    y0 = 2000  # base year for plotting

    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Define seasonal start/end for each contract month
    if cm == 3:  # H
        season_start = pd.Timestamp(year=year-1, month=4, day=1)
        season_end   = pd.Timestamp(year=year, month=3, day=12)
    elif cm == 5:  # K
        season_start = pd.Timestamp(year=year-1, month=6, day=1)
        season_end   = pd.Timestamp(year=year, month=5, day=12)
    elif cm == 7:  # N
        season_start = pd.Timestamp(year=year-1, month=8, day=1)
        season_end   = pd.Timestamp(year=year, month=7, day=12)
    elif cm == 9:  # U
        season_start = pd.Timestamp(year=year-1, month=11, day=1)
        season_end   = pd.Timestamp(year=year, month=9, day=30)
    elif cm == 12:  # Z
        season_start = pd.Timestamp(year=year, month=1, day=1)
        season_end   = pd.Timestamp(year=year, month=12, day=12)
    else:
        return pd.NaT

    # Compute virtual year offset
    if d < season_start:
        virtual_year = y0 - (season_start.year - d.year)
    else:
        virtual_year = y0 + (d.year - season_start.year)

    return safe_date(virtual_year, d.month, d.day)

# Apply to all prices
df_prices["virtual_date"] = df_prices.apply(assign_virtual_date_with_history, axis=1)
df_prices = df_prices.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
hrw_contract_ev = go.Figure()
contract_months = sorted(df_prices["contract_month"].unique())
colors = px.colors.qualitative.Plotly

for cm in contract_months:
    df_sub = df_prices[df_prices["contract_month"] == cm]
    years = sorted(df_sub["contract_year"].unique())
    season_color_map = {year: colors[i % len(colors)] for i, year in enumerate(years)}
    
    for year in years:
        df_year = df_sub[df_sub["contract_year"] == year].sort_values("virtual_date")
        hrw_contract_ev.add_trace(go.Scatter(
            x=df_year["virtual_date"],
            y=df_year["value"],
            mode="lines",
            name=f"{month_codes[cm]}{year}",
            line=dict(color=season_color_map[year], width=2)
        ))

# ---------------------------
# Buttons to filter by contract month
# ---------------------------
buttons = []
for cm in contract_months:
    visible = [f"{month_codes[cm]}" in trace.name for trace in hrw_contract_ev.data]
    buttons.append(dict(label=month_codes[cm], method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(hrw_contract_ev.data)}]))

# ---------------------------
# Layout
# ---------------------------
hrw_contract_ev.update_layout(
    title=dict(
        text="KC HRW Prices by Contract Month<br><sup>Seasonal Alignment</sup>",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b",dtick="M1",hoverformat='%d-%b' ),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

hrw_contract_ev.show()

# Export HTML if needed
hrw_contract_ev_html = hrw_contract_ev.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
kc_wheat=hrw[['date', 'value', 'contract_year', 'contract_month']]


month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}
spread_pairs = [(12, 3), (3, 5), (5, 7), (7, 9), (9, 12)]

pivot = kc_wheat.pivot_table(
    index="date",
    columns=["contract_year", "contract_month"],
    values="value"
)

spread_list = []
for near, far in spread_pairs:
    for year in kc_wheat['contract_year'].unique():
        try:
            near_price = pivot[(year, near)]
            far_price  = pivot[(year + (1 if near == 12 and far == 3 else 0), far)]
            spread_name = f"{month_codes[near]}{str(year)[-2:]}-{month_codes[far]}{str(year + (1 if near == 12 and far == 3 else 0))[-2:]}"
            tmp = pd.DataFrame({
                "date": near_price.index,
                "spread": near_price - far_price,
                "spread_type": f"{month_codes[near]}{month_codes[far]}",
                "season": year
            })
            spread_list.append(tmp)
        except KeyError:
            continue

spreads = pd.concat(spread_list).dropna()

# ---------------------------
# Filter last 9 months per spread/season
# ---------------------------
def keep_all_history_spreads_kc(df):
    df_list = []
    for stype in df['spread_type'].unique():
        df_sub = df[df['spread_type'] == stype]
        for season in df_sub['season'].unique():
            tmp = df_sub[df_sub['season'] == season].copy()
            if tmp.empty:
                continue

            # Define seasonal start and end (for reference, optional)
            if stype == "ZH":
                season_start = pd.Timestamp(season-1, 4, 1)
                season_end   = pd.Timestamp(season, 3, 31)
            elif stype == "HK":
                season_start = pd.Timestamp(season-1, 6, 1)
                season_end   = pd.Timestamp(season, 5, 31)
            elif stype == "KN":
                season_start = pd.Timestamp(season-1, 8, 1)
                season_end   = pd.Timestamp(season, 7, 31)
            elif stype == "NU":
                season_start = pd.Timestamp(season-1, 11, 1)
                season_end   = pd.Timestamp(season, 9, 30)
            elif stype == "UZ":
                season_start = pd.Timestamp(season, 1, 1)
                season_end   = pd.Timestamp(season, 12, 31)

            # Keep all data (no cutoff)
            tmp = tmp[(tmp['date'] >= tmp['date'].min()) & (tmp['date'] <= tmp['date'].max())]
            df_list.append(tmp)

    return pd.concat(df_list)

spreads = keep_all_history_spreads_kc(spreads)

# ---------------------------
# Assign virtual date for seasonal alignment (safe with leap years)
# ---------------------------
def assign_virtual_date_with_history_spreads_kc(row):
    d = row['date']
    stype = row['spread_type']
    season = row['season']  # the far contract year
    y0 = 2000  # base year for plotting
    
    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Define natural seasonal start/end
    if stype == "ZH":
        season_start = pd.Timestamp(season-1, 4, 1)
        season_end   = pd.Timestamp(season, 3, 31)
    elif stype == "HK":
        season_start = pd.Timestamp(season-1, 6, 1)
        season_end   = pd.Timestamp(season, 5, 31)
    elif stype == "KN":
        season_start = pd.Timestamp(season-1, 8, 1)
        season_end   = pd.Timestamp(season, 7, 31)
    elif stype == "NU":
        season_start = pd.Timestamp(season-1, 11, 1)
        season_end   = pd.Timestamp(season, 9, 30)
    elif stype == "UZ":
        season_start = pd.Timestamp(season, 1, 1)
        season_end   = pd.Timestamp(season, 12, 31)
    else:
        return pd.NaT

    # Compute virtual year offset
    if d < season_start:
        virtual_year = y0 - (season_start.year - d.year)  # negative offset
    else:
        virtual_year = y0 + (d.year - season_start.year)  # 0 or positive

    return safe_date(virtual_year, d.month, d.day)

# Apply to all spreads
spreads["virtual_date"] = spreads.apply(assign_virtual_date_with_history_spreads_kc, axis=1)
spreads = spreads.dropna(subset=["virtual_date"])


kc_cal_spread = go.Figure()
spread_types = spreads["spread_type"].unique()
colors = px.colors.qualitative.Plotly  # color palette

for stype in spread_types:
    df_sub = spreads[spreads["spread_type"] == stype]
    seasons = sorted(df_sub["season"].unique())
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}
    
    for season in seasons:
        season_data = df_sub[df_sub["season"] == season]
        season_data = season_data.sort_values("virtual_date")

        kc_cal_spread.add_trace(go.Scatter(
            x=season_data["virtual_date"],
            y=season_data["spread"],
            mode="lines",
            name=f"{stype} {season}",
            line=dict(color=season_color_map[season], width=2)
        ))

# ---------------------------
# Buttons to filter by spread type
# ---------------------------
buttons = []
for stype in spread_types:
    visible = [stype in trace.name for trace in kc_cal_spread.data]
    buttons.append(dict(label=stype, method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(kc_cal_spread.data)}]))

# ---------------------------
# Layout
# ---------------------------
kc_cal_spread.update_layout(
    title=dict(
        text="KC WHEAT Calendar Spreads<br><sup>From first contract's quote to expiry</sup>",
        x=0.5,  # center
        xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Spread (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

kc_cal_spread.show()


kc_cal_spread_html = kc_cal_spread.to_html(include_plotlyjs='cdn', full_html=True)

## SRW CBOT

In [0]:
srw= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-WHEAT-USDc-BU", period=f"{start_date}::{end_date}")
srw=srw[srw['observation']=='Settle']
srw=srw[['date','value','contract_year','contract_month']]
srw['day']=srw['date'].dt.day
srw['month']=srw['date'].dt.month


# CBOT month codes (if you want codes like Z25, H26, ...)
month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# Apply to your srw dataframe
srw['spot_year'], srw['spot_month'] = zip(*srw.apply(get_spot_contract, axis=1))
srw['spot_code'] = srw.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the contract rows that correspond to the rolling spot
spot_srw = srw[
    (srw['contract_year'] == srw['spot_year']) &
    (srw['contract_month'] == srw['spot_month'])
].copy()

spot_srw = spot_srw.sort_values('date')

cbot_srw_spot = go.Figure()

# Add the rolling spot contract line
cbot_srw_spot.add_trace(go.Scatter(
    x=spot_srw['date'],
    y=spot_srw['value'],
    mode='lines',
    name="CBOT SRW Spot",
    line=dict(color="blue", width=2)
))

# Add markers when the contract changes
roll_dates = spot_srw.loc[spot_srw['spot_code'].shift() != spot_srw['spot_code'], 'date']
roll_labels = spot_srw.loc[spot_srw['spot_code'].shift() != spot_srw['spot_code'], 'spot_code']

for d, code in zip(roll_dates, roll_labels):
    cbot_srw_spot.add_vline(x=d, line_dash="dash", line_color="gray")
    cbot_srw_spot.add_annotation(
        x=d, y=spot_srw['value'].max(),
        text=f"{code}",
        showarrow=False,
        yshift=20,
        font=dict(size=10, color="gray")
    )

# Layout settings
cbot_srw_spot.update_layout(
    title="CBOT SRW Rolling Spot Contract",
    xaxis_title="Date",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white"
)

cbot_srw_spot.show()
cbot_srw_spot_html = cbot_srw_spot.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
df_prices=srw[['date', 'value', 'contract_year', 'contract_month', 'day', 'month',
       'spot_year', 'spot_month']]

month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# Apply to all prices
df_prices["virtual_date"] = df_prices.apply(assign_virtual_date_with_history, axis=1)
df_prices = df_prices.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
srw_cbot_contract_ev = go.Figure()
contract_months = sorted(df_prices["contract_month"].unique())
colors = px.colors.qualitative.Plotly

for cm in contract_months:
    df_sub = df_prices[df_prices["contract_month"] == cm]
    years = sorted(df_sub["contract_year"].unique())
    season_color_map = {year: colors[i % len(colors)] for i, year in enumerate(years)}
    
    for year in years:
        df_year = df_sub[df_sub["contract_year"] == year].sort_values("virtual_date")
        srw_cbot_contract_ev.add_trace(go.Scatter(
            x=df_year["virtual_date"],
            y=df_year["value"],
            mode="lines",
            name=f"{month_codes[cm]}{year}",
            line=dict(color=season_color_map[year], width=2)
        ))

# ---------------------------
# Buttons to filter by contract month
# ---------------------------
buttons = []
for cm in contract_months:
    visible = [f"{month_codes[cm]}" in trace.name for trace in srw_cbot_contract_ev.data]
    buttons.append(dict(label=month_codes[cm], method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(srw_cbot_contract_ev.data)}]))

# ---------------------------
# Layout
# ---------------------------
srw_cbot_contract_ev.update_layout(
    title=dict(
        text="CBOT SRW Prices by Contract Month<br><sup>Seasonal Alignment</sup>",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b",dtick="M1",hoverformat='%d-%b' ),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

srw_cbot_contract_ev.show()

# Export HTML if needed
srw_cbot_contract_ev_html = srw_cbot_contract_ev.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
cbot_wheat=srw[['date', 'value', 'contract_year', 'contract_month']]


month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}
spread_pairs = [(12, 3), (3, 5), (5, 7), (7, 9), (9, 12)]

pivot = cbot_wheat.pivot_table(
    index="date",
    columns=["contract_year", "contract_month"],
    values="value"
)

spread_list = []
for near, far in spread_pairs:
    for year in cbot_wheat['contract_year'].unique():
        try:
            near_price = pivot[(year, near)]
            far_price  = pivot[(year + (1 if near == 12 and far == 3 else 0), far)]
            spread_name = f"{month_codes[near]}{str(year)[-2:]}-{month_codes[far]}{str(year + (1 if near == 12 and far == 3 else 0))[-2:]}"
            tmp = pd.DataFrame({
                "date": near_price.index,
                "spread": near_price - far_price,
                "spread_type": f"{month_codes[near]}{month_codes[far]}",
                "season": year
            })
            spread_list.append(tmp)
        except KeyError:
            continue

spreads = pd.concat(spread_list).dropna()

# ---------------------------
# Filter last 9 months per spread/season
# ---------------------------
spreads = keep_all_history_spreads_kc(spreads)

# Apply to all spreads
spreads["virtual_date"] = spreads.apply(assign_virtual_date_with_history_spreads_kc, axis=1)
spreads = spreads.dropna(subset=["virtual_date"])


cbot_cal_spread = go.Figure()
spread_types = spreads["spread_type"].unique()
colors = px.colors.qualitative.Plotly  # color palette

for stype in spread_types:
    df_sub = spreads[spreads["spread_type"] == stype]
    seasons = sorted(df_sub["season"].unique())
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}
    
    for season in seasons:
        season_data = df_sub[df_sub["season"] == season]
        season_data = season_data.sort_values("virtual_date")

        cbot_cal_spread.add_trace(go.Scatter(
            x=season_data["virtual_date"],
            y=season_data["spread"],
            mode="lines",
            name=f"{stype} {season}",
            line=dict(color=season_color_map[season], width=2)
        ))

# ---------------------------
# Buttons to filter by spread type
# ---------------------------
buttons = []
for stype in spread_types:
    visible = [stype in trace.name for trace in cbot_cal_spread.data]
    buttons.append(dict(label=stype, method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(cbot_cal_spread.data)}]))

# ---------------------------
# Layout
# ---------------------------
cbot_cal_spread.update_layout(
    title=dict(
        text="CBOT SRW WHEAT Calendar Spreads<br><sup>From first contract's quote to expiry</sup>",
        x=0.5,  # center
        xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Spread (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

cbot_cal_spread.show()


cbot_cal_spread_html = cbot_cal_spread.to_html(include_plotlyjs='cdn', full_html=True)




# KC VS CBOT SPREADS

In [0]:
hrw=zema.get_curve(curve="P-FUTURE-CBOT-INPUT-HRW Wheat-USDc-BU", period=f"{start_date}::{end_date}")
hrw=hrw[hrw['observation']=='Settle']
hrw=hrw[['date','value','contract_year','contract_month']]
hrw.rename(columns={'value': 'kc_wheat'}, inplace=True)

srw= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-WHEAT-USDc-BU", period=f"{start_date}::{end_date}")
srw=srw[srw['observation']=='Settle']
srw=srw[['date','value','contract_year','contract_month']]
srw.rename(columns={'value': 'cbot_wheat'}, inplace=True)

spread_wheat=pd.merge(hrw,srw,on=['date','contract_year','contract_month'],how='inner')
spread_wheat['day']=spread_wheat['date'].dt.day
spread_wheat['month']=spread_wheat['date'].dt.month
spread_wheat['spread']=spread_wheat['kc_wheat']-spread_wheat['cbot_wheat']

# Apply to your srw dataframe
spread_wheat['spot_year'], spread_wheat['spot_month'] = zip(*spread_wheat.apply(get_spot_contract, axis=1))
spread_wheat['spot_code'] = spread_wheat.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the contract rows that correspond to the rolling spot
spot_spread_wheat = spread_wheat[
    (spread_wheat['contract_year'] == spread_wheat['spot_year']) &
    (spread_wheat['contract_month'] == spread_wheat['spot_month'])
].copy()


df_prices=spread_wheat[['date', 'spread', 'contract_year', 'contract_month', 'day', 'month',
       'spot_year', 'spot_month']]

month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# Apply to all prices
df_prices["virtual_date"] = df_prices.apply(assign_virtual_date_with_history, axis=1)
df_prices = df_prices.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
wheat_spreads_contract_ev = go.Figure()
contract_months = sorted(df_prices["contract_month"].unique())
colors = px.colors.qualitative.Plotly

for cm in contract_months:
    df_sub = df_prices[df_prices["contract_month"] == cm]
    years = sorted(df_sub["contract_year"].unique())
    season_color_map = {year: colors[i % len(colors)] for i, year in enumerate(years)}
    
    for year in years:
        df_year = df_sub[df_sub["contract_year"] == year].sort_values("virtual_date")
        wheat_spreads_contract_ev.add_trace(go.Scatter(
            x=df_year["virtual_date"],
            y=df_year["spread"],
            mode="lines",
            name=f"{month_codes[cm]}{year}",
            line=dict(color=season_color_map[year], width=2)
        ))

# ---------------------------
# Buttons to filter by contract month
# ---------------------------
buttons = []
for cm in contract_months:
    visible = [f"{month_codes[cm]}" in trace.name for trace in wheat_spreads_contract_ev.data]
    buttons.append(dict(label=month_codes[cm], method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(wheat_spreads_contract_ev.data)}]))

# ---------------------------
# Layout
# ---------------------------
wheat_spreads_contract_ev.update_layout(
    title=dict(
        text="KC Wheat - CBOT Wheat Prices by Contract Month<br><sup>Seasonal Alignment</sup>",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b",dtick="M1",hoverformat='%d-%b' ),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

wheat_spreads_contract_ev.show()

# Export HTML if needed
wheat_spreads_contract_ev_html = wheat_spreads_contract_ev.to_html(include_plotlyjs='cdn', full_html=True)


#CBOT WHEAT VS CBOT CORN

In [0]:
cbot_corn= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-CORN-USDc-BU", period=f"{start_date}::{end_date}")
cbot_corn=cbot_corn[cbot_corn['observation']=='Settle']
cbot_corn=cbot_corn[['date','value','contract_year','contract_month']]
cbot_corn.rename(columns={'value': 'cbot_corn'}, inplace=True)
srw= zema.get_curve(curve="P-FUTURE-CBOT-INPUT-WHEAT-USDc-BU", period=f"{start_date}::{end_date}")
srw=srw[srw['observation']=='Settle']
srw=srw[['date','value','contract_year','contract_month']]
srw.rename(columns={'value': 'cbot_wheat'}, inplace=True)
cbot_prices=pd.merge(cbot_corn,srw,on=['date','contract_year','contract_month'],how='inner')
cbot_prices['day']=cbot_prices['date'].dt.day
cbot_prices['month']=cbot_prices['date'].dt.month
cbot_prices['spread']=cbot_prices['cbot_wheat']-cbot_prices['cbot_corn']

# Apply to your srw dataframe
cbot_prices['spot_year'], cbot_prices['spot_month'] = zip(*cbot_prices.apply(get_spot_contract, axis=1))
cbot_prices['spot_code'] = cbot_prices.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the contract rows that correspond to the rolling spot
spot_cbot_prices = cbot_prices[
    (cbot_prices['contract_year'] == cbot_prices['spot_year']) &
    (cbot_prices['contract_month'] == cbot_prices['spot_month'])
].copy()


df_prices=cbot_prices[['date', 'spread', 'contract_year', 'contract_month', 'day', 'month',
       'spot_year', 'spot_month']]

month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# Apply to all prices
df_prices["virtual_date"] = df_prices.apply(assign_virtual_date_with_history, axis=1)
df_prices = df_prices.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
cbot_spreads_contract_ev = go.Figure()
contract_months = sorted(df_prices["contract_month"].unique())
colors = px.colors.qualitative.Plotly

for cm in contract_months:
    df_sub = df_prices[df_prices["contract_month"] == cm]
    years = sorted(df_sub["contract_year"].unique())
    season_color_map = {year: colors[i % len(colors)] for i, year in enumerate(years)}
    
    for year in years:
        df_year = df_sub[df_sub["contract_year"] == year].sort_values("virtual_date")
        cbot_spreads_contract_ev.add_trace(go.Scatter(
            x=df_year["virtual_date"],
            y=df_year["spread"],
            mode="lines",
            name=f"{month_codes[cm]}{year}",
            line=dict(color=season_color_map[year], width=2)
        ))

# ---------------------------
# Buttons to filter by contract month
# ---------------------------
buttons = []
for cm in contract_months:
    visible = [f"{month_codes[cm]}" in trace.name for trace in cbot_spreads_contract_ev.data]
    buttons.append(dict(label=month_codes[cm], method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(cbot_spreads_contract_ev.data)}]))

# ---------------------------
# Layout
# ---------------------------
cbot_spreads_contract_ev.update_layout(
    title=dict(
        text="CBOT WHEAT - CBOT CORN Prices by Contract Month<br><sup>Seasonal Alignment</sup>",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b",dtick="M1",hoverformat='%d-%b' ),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

cbot_spreads_contract_ev.show()

# Export HTML if needed
cbot_spreads_contract_ev_html = cbot_spreads_contract_ev.to_html(include_plotlyjs='cdn', full_html=True)


# MATIF SPOT

In [0]:
matif= zema.get_curve(curve="P-FUTURE-ENXT-INPUT-WHEAT-EUR-MT", period=f"{start_date}::{end_date}")
matif=matif[matif['observation']=='Settle']
matif.rename(columns={'value': 'matif_eur'}, inplace=True)
matif=matif[['date','matif_eur','contract_year','contract_month']]



In [0]:
import calendar

eurusd= zema.get_curve(curve='FX_GPL_FWD_EURUSD', period=f"{start_date}::{end_date}")

def recreate_full_date(contract_month, contract_year):
    # Define the base date and base number
    base_date = datetime(2024, 12, 2)
    base_number = 4336

    # Calculate the number of days offset
    delta_days = contract_month - base_number
    calculated_date = base_date + timedelta(days=delta_days)

    # Extract the day of the month and month number
    day_of_month = calculated_date.day
    month_number = calculated_date.month

    # Get the last day of the target month
    last_day_of_month = calendar.monthrange(contract_year, month_number)[1]

    # Adjust the day if it exceeds the last day of the month
    if day_of_month > last_day_of_month:
        day_of_month = last_day_of_month

    # Use the provided year from the contract_year column
    return datetime(contract_year, month_number, day_of_month)
  

# Define a base date for the mapping
base_date = datetime(2024, 12, 2)
base_number = 4336

# Apply the mapping function to the column
eurusd['Contract Month Date'] = eurusd.apply(
    lambda row: recreate_full_date(row['contract_month'], row['contract_year']), axis=1
)

#eurusd=eurusd[['date','value','contract_year','Contract Month Date']]
eurusd['contract_month']=eurusd['Contract Month Date'].dt.month
eurusd.rename(columns={'value': 'eurusd'}, inplace=True)

# Group by the required columns and calculate the average eurusd
eurusd = eurusd.groupby(['date', 'contract_year', 'contract_month'], as_index=False)['eurusd'].mean()

matif_usd = pd.merge(
    matif,
    eurusd,
    on=['date', 'contract_year', 'contract_month'],
    how='inner'  # You can also use 'outer', 'left', or 'right' depending on your needs
)



In [0]:
matif_usd['matif_usd']=matif_usd['matif_eur']*matif_usd['eurusd']*(100/36.7437)
matif_usd=matif_usd[['date','matif_usd','contract_year','contract_month']]

In [0]:
def get_spot_contract_matif(row):
    d = row['date']
    y = d.year
    m = d.month
    day = d.day

    # Sep 20 -> Dec 12  => Dec of current year
    if (m == 9 and day >= 13) or (m in [10, 11]) or (m == 12 and day <= 12):
        return (y, 12)

    # Dec 20 -> Dec 31 => March of NEXT year (roll happens at year end)
    if m == 12 and day >= 13:
        return (y + 1, 3)

    # Jan 1 -> Mar 12  => March of CURRENT year
    if (m in [1, 2]) or (m == 3 and day <= 12):
        return (y, 3)

    # Mar 20 -> May 12 => May of current year
    if (m == 3 and day >= 13) or (m == 4) or (m == 5 and day <= 12):
        return (y, 5)

    # May 13 -> Sep 12 => September of current year
    if (m == 5 and day >= 13) or (m == 8) or (m == 6)or (m == 7) or (m == 9 and day <= 12):
        return (y, 9)

    # safety fallback (shouldn't be hit with your rules)
    return (y, m)


# Apply to your hrw dataframe
matif_usd['spot_year'], matif_usd['spot_month'] = zip(*matif_usd.apply(get_spot_contract_matif, axis=1))
matif_usd['spot_code'] = matif_usd.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the contract rows that correspond to the rolling spot
spot_matif_usd = matif_usd[
    (matif_usd['contract_year'] == matif_usd['spot_year']) &
    (matif_usd['contract_month'] == matif_usd['spot_month'])
].copy()

spot_matif_usd = spot_matif_usd.sort_values('date')


In [0]:
matif_usd_spot = go.Figure()

# Add the rolling spot contract line
matif_usd_spot.add_trace(go.Scatter(
    x=spot_matif_usd['date'],
    y=spot_matif_usd['matif_usd'],
    mode='lines',
    name="MATIF USD Spot",
    line=dict(color="blue", width=2)
))

# Add markers when the contract changes
roll_dates = spot_matif_usd.loc[spot_matif_usd['spot_code'].shift() != spot_matif_usd['spot_code'], 'date']
roll_labels = spot_matif_usd.loc[spot_matif_usd['spot_code'].shift() != spot_matif_usd['spot_code'], 'spot_code']

for d, code in zip(roll_dates, roll_labels):
    matif_usd_spot.add_vline(x=d, line_dash="dash", line_color="gray")
    matif_usd_spot.add_annotation(
        x=d, y=spot_matif_usd['matif_usd'].max(),
        text=f"{code}",
        showarrow=False,
        yshift=20,
        font=dict(size=10, color="gray")
    )

# Layout settings
matif_usd_spot.update_layout(
    title="Matif USD Rolling Spot Contract",
    xaxis_title="Date",
    yaxis_title="Price (USDc/MT)",
    hovermode="x unified",
    template="plotly_white"
)

matif_usd_spot.show()
matif_usd_spot_html = matif_usd_spot.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
matif_usd['day']=matif_usd['date'].dt.day
matif_usd['month']=matif_usd['date'].dt.month
df_prices_matif=matif_usd[['date', 'matif_usd', 'contract_year', 'contract_month', 'day', 'month',
       'spot_year', 'spot_month']]

month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}

# ---------------------------
# Assign virtual date for seasonal alignment (like spreads)
# ---------------------------
def assign_virtual_date_with_history_matif(row):
    d = row['date']
    cm = row['contract_month']
    year = row['contract_year']  # the contract year
    y0 = 2000  # base year for plotting

    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Define seasonal start/end for each contract month
    if cm == 3:  # H
        season_start = pd.Timestamp(year=year-1, month=4, day=1)
        season_end   = pd.Timestamp(year=year, month=3, day=12)
    elif cm == 5:  # K
        season_start = pd.Timestamp(year=year-1, month=6, day=1)
        season_end   = pd.Timestamp(year=year, month=5, day=12)
    elif cm == 9:  # U
        season_start = pd.Timestamp(year=year-1, month=11, day=1)
        season_end   = pd.Timestamp(year=year, month=9, day=30)
    elif cm == 12:  # Z
        season_start = pd.Timestamp(year=year, month=1, day=1)
        season_end   = pd.Timestamp(year=year, month=12, day=12)
    else:
        return pd.NaT

    # Compute virtual year offset
    if d < season_start:
        virtual_year = y0 - (season_start.year - d.year)
    else:
        virtual_year = y0 + (d.year - season_start.year)

    return safe_date(virtual_year, d.month, d.day)

# Apply to all prices
df_prices_matif["virtual_date"] = df_prices_matif.apply(assign_virtual_date_with_history_matif, axis=1)
df_prices_matif = df_prices_matif.dropna(subset=["virtual_date"])

# ---------------------------
# Plot
# ---------------------------
matif_contract_ev = go.Figure()
contract_months = sorted(df_prices_matif["contract_month"].unique())
colors = px.colors.qualitative.Plotly

for cm in contract_months:
    df_sub = df_prices_matif[df_prices_matif["contract_month"] == cm]
    years = sorted(df_sub["contract_year"].unique())
    season_color_map = {year: colors[i % len(colors)] for i, year in enumerate(years)}
    
    for year in years:
        df_year = df_sub[df_sub["contract_year"] == year].sort_values("virtual_date")
        matif_contract_ev.add_trace(go.Scatter(
            x=df_year["virtual_date"],
            y=df_year["matif_usd"],
            mode="lines",
            name=f"{month_codes[cm]}{year}",
            line=dict(color=season_color_map[year], width=2)
        ))

# ---------------------------
# Buttons to filter by contract month
# ---------------------------
buttons = []
for cm in contract_months:
    visible = [f"{month_codes[cm]}" in trace.name for trace in matif_contract_ev.data]
    buttons.append(dict(label=month_codes[cm], method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(matif_contract_ev.data)}]))

# ---------------------------
# Layout
# ---------------------------
matif_contract_ev.update_layout(
    title=dict(
        text="MATIF USD Prices by Contract Month<br><sup>Seasonal Alignment</sup>",
        x=0.5, xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Price (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b",dtick="M1",hoverformat='%d-%b' ),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

matif_contract_ev.show()

# Export HTML if needed
matif_contract_ev_html = matif_contract_ev.to_html(include_plotlyjs='cdn', full_html=True)


##MATIF CALENDAR SPREADS

In [0]:
matif_wheat=matif_usd[['date', 'matif_usd', 'contract_year', 'contract_month']]


month_codes = {3: "H", 5: "K", 7: "N", 9: "U", 12: "Z"}
spread_pairs = [(12, 3), (3, 5), (5, 9), (9, 12)]

pivot = matif_wheat.pivot_table(
    index="date",
    columns=["contract_year", "contract_month"],
    values="matif_usd"
)

spread_list = []
for near, far in spread_pairs:
    for year in matif_wheat['contract_year'].unique():
        try:
            near_price = pivot[(year, near)]
            far_price  = pivot[(year + (1 if near == 12 and far == 3 else 0), far)]
            spread_name = f"{month_codes[near]}{str(year)[-2:]}-{month_codes[far]}{str(year + (1 if near == 12 and far == 3 else 0))[-2:]}"
            tmp = pd.DataFrame({
                "date": near_price.index,
                "spread": near_price - far_price,
                "spread_type": f"{month_codes[near]}{month_codes[far]}",
                "season": year
            })
            spread_list.append(tmp)
        except KeyError:
            continue

spreads = pd.concat(spread_list).dropna()

# ---------------------------
# Filter last 9 months per spread/season
# ---------------------------
def keep_all_history_spreads_matif(df):
    df_list = []
    for stype in df['spread_type'].unique():
        df_sub = df[df['spread_type'] == stype]
        for season in df_sub['season'].unique():
            tmp = df_sub[df_sub['season'] == season].copy()
            if tmp.empty:
                continue

            # Define seasonal start and end (for reference, optional)
            if stype == "ZH":
                season_start = pd.Timestamp(season-1, 4, 1)
                season_end   = pd.Timestamp(season, 3, 31)
            elif stype == "HK":
                season_start = pd.Timestamp(season-1, 6, 1)
                season_end   = pd.Timestamp(season, 5, 31)
            elif stype == "KU":
                season_start = pd.Timestamp(season-1, 8, 1)
                season_end   = pd.Timestamp(season, 9, 30)
            elif stype == "UZ":
                season_start = pd.Timestamp(season, 1, 1)
                season_end   = pd.Timestamp(season, 12, 31)

            # Keep all data (no cutoff)
            tmp = tmp[(tmp['date'] >= tmp['date'].min()) & (tmp['date'] <= tmp['date'].max())]
            df_list.append(tmp)

    return pd.concat(df_list)

spreads = keep_all_history_spreads_matif(spreads)

# ---------------------------
# Assign virtual date for seasonal alignment (safe with leap years)
# ---------------------------
def assign_virtual_date_with_history_spreads_matif(row):
    d = row['date']
    stype = row['spread_type']
    season = row['season']  # the far contract year
    y0 = 2000  # base year for plotting
    
    def safe_date(y, m, day):
        try:
            return pd.Timestamp(year=y, month=m, day=day)
        except ValueError:
            if m == 2 and day == 29:
                return pd.Timestamp(year=y, month=2, day=28)
            return pd.NaT

    # Define natural seasonal start/end
    if stype == "ZH":
        season_start = pd.Timestamp(season-1, 4, 1)
        season_end   = pd.Timestamp(season, 3, 31)
    elif stype == "HK":
        season_start = pd.Timestamp(season-1, 6, 1)
        season_end   = pd.Timestamp(season, 5, 31)
    elif stype == "KU":
        season_start = pd.Timestamp(season-1, 8, 1)
        season_end   = pd.Timestamp(season, 9, 30)
    elif stype == "UZ":
        season_start = pd.Timestamp(season, 1, 1)
        season_end   = pd.Timestamp(season, 12, 31)
    else:
        return pd.NaT

    # Compute virtual year offset
    if d < season_start:
        virtual_year = y0 - (season_start.year - d.year)  # negative offset
    else:
        virtual_year = y0 + (d.year - season_start.year)  # 0 or positive

    return safe_date(virtual_year, d.month, d.day)

# Apply to all spreads
spreads["virtual_date"] = spreads.apply(assign_virtual_date_with_history_spreads_matif, axis=1)
spreads = spreads.dropna(subset=["virtual_date"])


matif_cal_spread = go.Figure()
spread_types = spreads["spread_type"].unique()
colors = px.colors.qualitative.Plotly  # color palette

for stype in spread_types:
    df_sub = spreads[spreads["spread_type"] == stype]
    seasons = sorted(df_sub["season"].unique())
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}
    
    for season in seasons:
        season_data = df_sub[df_sub["season"] == season]
        season_data = season_data.sort_values("virtual_date")

        matif_cal_spread.add_trace(go.Scatter(
            x=season_data["virtual_date"],
            y=season_data["spread"],
            mode="lines",
            name=f"{stype} {season}",
            line=dict(color=season_color_map[season], width=2)
        ))

# ---------------------------
# Buttons to filter by spread type
# ---------------------------
buttons = []
for stype in spread_types:
    visible = [stype in trace.name for trace in matif_cal_spread.data]
    buttons.append(dict(label=stype, method="update", args=[{"visible": visible}]))

# Add ALL button
buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True]*len(matif_cal_spread.data)}]))

# ---------------------------
# Layout
# ---------------------------
matif_cal_spread.update_layout(
    title=dict(
        text="MATIF WHEAT Calendar Spreads<br><sup>From first contract's quote to expiry</sup>",
        x=0.5,  # center
        xanchor="center"
    ),
    autosize=True,
    xaxis_title="Seasonal Timeline",
    yaxis_title="Spread (USDc/bu)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(tickformat="%b"),
    updatemenus=[dict(
        type="buttons",
        direction="left",
        x=0,
        y=1.15,
        xanchor="left",
        yanchor="top",
        buttons=buttons,
        showactive=True
    )]
)

matif_cal_spread.show()


matif_cal_spread_html = matif_cal_spread.to_html(include_plotlyjs='cdn', full_html=True)

# FX EURUSD

In [0]:
month_codes = {
    1: "F",  # January
    2: "G",  # February
    3: "H",  # March
    4: "J",  # April
    5: "K",  # May
    6: "M",  # June
    7: "N",  # July
    8: "Q",  # August
    9: "U",  # September
    10: "V", # October
    11: "X", # November
    12: "Z"  # December
}

def get_spot_contract_fx(row):
    d = row['date']
    y = d.year
    m = d.month
    day = d.day

    # If after the 12th, roll to next month
    if day >= 13:
        if m == 12:
            # December -> January next year
            return (y + 1, 1)
        else:
            return (y, m + 1)
    else:
        # 1st to 12th stays in current month
        return (y, m)


# Apply to your dataframe
eurusd['spot_year'], eurusd['spot_month'] = zip(*eurusd.apply(get_spot_contract_fx, axis=1))
eurusd['spot_code'] = eurusd.apply(lambda r: month_codes[r['spot_month']] + str(r['spot_year'])[-2:], axis=1)

# Filter to keep only the rolling spot contract
spot_eurusd = eurusd[
    (eurusd['contract_year'] == eurusd['spot_year']) &
    (eurusd['contract_month'] == eurusd['spot_month'])
].copy()

spot_eurusd = spot_eurusd.sort_values('date')
spot_eurusd = spot_eurusd[spot_eurusd['eurusd'] > 0.8]


eurusd_spot = go.Figure()

# Add the rolling spot contract line
eurusd_spot.add_trace(go.Scatter(
    x=spot_eurusd['date'],
    y=spot_eurusd['eurusd'],
    mode='lines',
    name="EURUSD Spot",
    line=dict(color="blue", width=2)
))

# Add markers when the contract changes
roll_dates = spot_eurusd.loc[spot_eurusd['spot_code'].shift() != spot_eurusd['spot_code'], 'date']
roll_labels = spot_eurusd.loc[spot_eurusd['spot_code'].shift() != spot_eurusd['spot_code'], 'spot_code']


# Layout settings
eurusd_spot.update_layout(
    title="EURUSD Rolling Spot Contract",
    xaxis_title="Date",
    yaxis_title="EURUSD",
    hovermode="x unified",
    template="plotly_white"
)

eurusd_spot.show()
eurusd_spot_html = eurusd_spot.to_html(include_plotlyjs='cdn', full_html=True)


# FOB PRICES COMPARISON

In [0]:
import pandas as pd

def get_spot_curve(curve_name: str, start_date: str, end_date: str):
    """
    Fetches a curve from ZEMA and processes it to return the spot curve DataFrame.
    
    Parameters
    ----------
    curve_name : str
        The curve name in ZEMA.
    start_date : str or datetime
        Start date for the query (e.g. '2020-01-01').
    end_date : str or datetime
        End date for the query (e.g. '2025-01-01').

    Returns
    -------
    pd.DataFrame
        DataFrame with ['date', 'value', 'contract_year', 'contract_month',
        'virtual_date', 'contract_date'] for the spot curve.
    """
    
    # Fetch curve
    df = zema.get_curve(curve=curve_name, period=f"{start_date}::{end_date}")
    
    # Keep only "Last" observations
    df = df[df['observation'] == 'Last']
    
    # Keep relevant columns
    df = df[['date', 'value', 'contract_year', 'contract_month']].copy()
    
    # Extract calendar parts
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    
    # Create "virtual date" (aligning all years to 2000 for seasonality)
    df['virtual_date'] = pd.to_datetime(
        {'year': 2000, 'month': df['month'], 'day': df['day']},
        errors='coerce'
    )
    
    # Drop invalid dates
    df = df.dropna(subset=['virtual_date'])
    
    # Ensure datetime types
    df['date'] = pd.to_datetime(df['date'])
    df['virtual_date'] = pd.to_datetime(df['virtual_date'])
    
    # Create contract date (first day of contract month/year)
    df['contract_date'] = pd.to_datetime(dict(
        year=df['contract_year'],
        month=df['contract_month'],
        day=1
    ))
    
    # For each real date, pick the earliest available contract
    spot_df = (
        df.sort_values(['date', 'contract_date'])
          .groupby('date')
          .first()
          .reset_index()
    )
    
    return spot_df



def plot_flat_premium(flat_df=None, prem_df=None, title="Flat and Premium", width=1800, height=700, colors=None):
    """
    Plots Flat (FOB) and Premium values by season with interactive filtering.
    Works even if only one of flat_df or prem_df is provided.
    
    Parameters
    ----------
    flat_df : pd.DataFrame, optional
        DataFrame containing 'virtual_date', 'flat', and 'contract_year'.
    prem_df : pd.DataFrame, optional
        DataFrame containing 'virtual_date', 'premium', and 'contract_year'.
    title : str, optional
        Plot title (default: "Flat and Premium").
    width : int, optional
        Plot width in pixels (default: 1800).
    height : int, optional
        Plot height in pixels (default: 700).
    colors : list, optional
        List of colors to cycle through. Defaults to Plotly qualitative palette.
    """

    # Default colors
    if colors is None:
        colors = px.colors.qualitative.Plotly

    # Initialize empty figure
    fig = go.Figure()

    # Collect season list from both dfs
    season_list = set()
    if flat_df is not None and not flat_df.empty:
        flat_df = flat_df.copy()
        flat_df['season'] = flat_df['contract_year'].astype(str)
        season_list.update(flat_df['season'].unique())
    if prem_df is not None and not prem_df.empty:
        prem_df = prem_df.copy()
        prem_df['season'] = prem_df['contract_year'].astype(str)
        season_list.update(prem_df['season'].unique())

    season_list = sorted(season_list)
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(season_list)}

    # Add FOB traces if flat_df exists
    if flat_df is not None and not flat_df.empty:
        for season in season_list:
            season_data = flat_df[flat_df['season'] == season].sort_values('virtual_date')
            if not season_data.empty:
                fig.add_trace(go.Scatter(
                    x=season_data['virtual_date'],
                    y=season_data['flat'],
                    mode='lines',
                    name=f"FOB {season}",
                    line=dict(color=season_color_map[season], width=2, dash='solid'),
                    yaxis="y1"
                ))

    # Add Premium traces if prem_df exists
    if prem_df is not None and not prem_df.empty:
        for season in season_list:
            season_data = prem_df[prem_df['season'] == season].sort_values('virtual_date')
            if not season_data.empty:
                fig.add_trace(go.Scatter(
                    x=season_data['virtual_date'],
                    y=season_data['premium'],
                    mode='lines',
                    name=f"Premium {season}",
                    line=dict(color=season_color_map[season], width=2, dash='dash'),
                    yaxis="y2"
                ))

    # Layout with dual y-axes
    fig.update_layout(
        title=title,
        xaxis=dict(
            tickformat='%b',
            dtick="M1",
            hoverformat='%d-%b'
        ),
        yaxis=dict(title='FOB Spot Price'),
        yaxis2=dict(
            title='Premium',
            overlaying='y',
            side='right',
            showgrid=False
        ),
        template='plotly_white',
        height=height,
        width=width,
        hovermode='x unified'
    )

    # Add Buttons for filtering by Season
    if season_list:
        buttons = []
        for season in season_list:
            visible = [season in trace.name for trace in fig.data]
            buttons.append(dict(
                label=season,
                method="update",
                args=[{"visible": visible}]
            ))
        buttons.insert(0, dict(
            label="ALL",
            method="update",
            args=[{"visible": [True] * len(fig.data)}]
        ))

        fig.update_layout(
            updatemenus=[dict(
                type="buttons",
                direction="left",
                x=0,
                y=1.05,
                xanchor="left",
                yanchor="top",
                buttons=buttons,
                showactive=True
            )]
        )

    return fig


## UPR

In [0]:
wheat_factor_bu_to_mt=36.7437

In [0]:
wheat_upr_spot_flat=get_spot_curve('P-CASH-LDC-INPUT-FLAT-WHEAT-AR-FOB Up River-11.5-USD-MT',start_date,end_date)
wheat_upr_spot_flat.rename(columns={'value':'flat'},inplace=True)
wheat_upr_spot_flat['season'] = "FOB " + wheat_upr_spot_flat['year'].astype(str)
wheat_upr_spot_flat['origin']='UPR 11.5'


wheat_texas_spot_flat=get_spot_curve('P-CASH-LDC-CALC-FLAT-WHEAT-US-FOB-FBVTX-HRW 11.0-USDc-Bu',start_date,end_date)
wheat_texas_spot_flat.rename(columns={'value':'flat'},inplace=True)
wheat_texas_spot_flat['flat']=wheat_texas_spot_flat['flat']*wheat_factor_bu_to_mt/100
wheat_texas_spot_flat['season'] = "FOB " + wheat_texas_spot_flat['year'].astype(str)
wheat_texas_spot_flat['origin']='Texas 11'

wheat_ukraine_spot_flat=get_spot_curve('P-CASH-LDC-INPUT-FLAT-WHEAT-UA-FOB Nikolaev-11.5-USD-MT',start_date,end_date)
wheat_ukraine_spot_flat.rename(columns={'value':'flat'},inplace=True)
wheat_ukraine_spot_flat['season'] = "FOB " + wheat_ukraine_spot_flat['year'].astype(str)
wheat_ukraine_spot_flat['origin']='Nikolaev 11.5'

wheat_novo_spot_flat=get_spot_curve('P-CASH-LDC-INPUT-FLAT-WHEAT-RU-FOB Novo-12.5-USD-MT',start_date,end_date)
wheat_novo_spot_flat.rename(columns={'value':'flat'},inplace=True)
wheat_novo_spot_flat['season'] = "FOB " + wheat_novo_spot_flat['year'].astype(str)
wheat_novo_spot_flat['origin']='Novo 12.5'


wheat_apw_spot_flat=get_spot_curve('P-CASH-LDC-INPUT-FLAT-WHEAT-AU-FOB KWINANA-APW1-USD-MT',start_date,end_date)
wheat_apw_spot_flat.rename(columns={'value':'flat'},inplace=True)
wheat_apw_spot_flat['season'] = "FOB " + wheat_apw_spot_flat['year'].astype(str)
wheat_apw_spot_flat['origin']='APW KWINANA'
wheat_apw_spot_flat


In [0]:
wheat_dfs=pd.concat([wheat_upr_spot_flat,wheat_texas_spot_flat,wheat_ukraine_spot_flat,wheat_novo_spot_flat,wheat_apw_spot_flat])
wheat_dfs=wheat_dfs[wheat_dfs['flat']!=0]

In [0]:

def plot_flat_origins(flat_df, title="Flat FOB by Origin", width=1800, height=700, colors=None):
    """
    Plots Flat (FOB) values by origin and season with interactive filtering.
    """

    if colors is None:
        colors = px.colors.qualitative.Plotly

    flat_df = flat_df.copy()
    flat_df['season'] = flat_df['contract_year'].astype(str)
    flat_df = flat_df.sort_values("virtual_date")

    origins = sorted(flat_df['origin'].unique())
    seasons = sorted(flat_df['season'].unique())

    # Assign colors by season
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(seasons)}

    # Assign line styles by origin
    dash_styles = ["solid", "dot", "dash", "longdash", "dashdot", "longdashdot"]
    origin_style_map = {origin: dash_styles[i % len(dash_styles)] for i, origin in enumerate(origins)}

    fig = go.Figure()

    # Add one trace per (origin, season)
    for origin in origins:
        for season in seasons:
            subset = flat_df[(flat_df['origin'] == origin) & (flat_df['season'] == season)]
            if not subset.empty:
                fig.add_trace(go.Scatter(
                    x=subset['virtual_date'],
                    y=subset['flat'],
                    mode='lines',
                    name=f"{origin} {season}",
                    line=dict(
                        color=season_color_map[season],
                        dash=origin_style_map[origin],  # style by origin
                        width=2
                    ),
                ))

    # Layout
    fig.update_layout(
        title=title,
        xaxis=dict(
            tickformat='%b',
            dtick="M1",
            hoverformat='%d-%b'
        ),
        yaxis=dict(title='FOB Price'),
        template='plotly_white',
        height=height,
        width=width,
        hovermode='x unified'
    )

    # Build filter buttons
    buttons = []
    buttons.append(dict(label="ALL", method="update", args=[{"visible": [True] * len(fig.data)}]))

    # Buttons by origin
    for origin in origins:
        visible = [origin in trace.name for trace in fig.data]
        buttons.append(dict(label=f"{origin} (all years)", method="update", args=[{"visible": visible}]))

    # Buttons by season
    for season in seasons:
        visible = [season in trace.name for trace in fig.data]
        buttons.append(dict(label=f"{season}", method="update", args=[{"visible": visible}]))

    # Place buttons horizontally
    fig.update_layout(
        updatemenus=[dict(
            type="buttons",
            direction="right",
            x=0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True
        )]
    )

    return fig



In [0]:
spot_chart = plot_flat_origins(wheat_dfs)
spot_chart.show()

spot_chart_html = spot_chart.to_html(include_plotlyjs='cdn', full_html=True)

# MATBA VS HISTORY

In [0]:
wheat_matba= zema.get_curve(curve="P-FUTURE-MATBA-INPUT-WHEAT-ROS-USD-MT", period=f"{start_date}::{end_date}")
wheat_matba=wheat_matba[wheat_matba['observation']=='Settle']

wheat_matba=wheat_matba[['date','value','contract_year','contract_month']]
wheat_matba['day']=wheat_matba['date'].dt.day
wheat_matba['month']=wheat_matba['date'].dt.month
wheat_matba['year']=wheat_matba['date'].dt.year
wheat_matba['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': wheat_matba['month'], 'day': wheat_matba['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)
# Drop rows with invalid virtual dates
wheat_matba = wheat_matba.dropna(subset=['virtual_date'])
wheat_matba

# MATBA WHEAT MARCH JULY AND DECEMBER FLAT

## MARCH

In [0]:
wheat_matba_march=wheat_matba[wheat_matba['contract_month']==3]

wheat_matba_march['Season'] = 'March'+ wheat_matba_march['contract_year'].astype(str)
  
def adjust_virtual_date_wheat_mar(row):
    if row['virtual_date'].month in [1, 2,3]:
        # Subtract 1 year from the year if month is November or marchember
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or marchember
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
wheat_matba_march['virtual_date'] = wheat_matba_march.apply(adjust_virtual_date_wheat_mar, axis=1)
wheat_matba_march = wheat_matba_march[wheat_matba_march['value'] != 0]

wheat_matba_march = wheat_matba_march[
    (wheat_matba_march['date'].dt.month.isin([1, 2, 3]) & (wheat_matba_march['date'].dt.year == wheat_matba_march['contract_year'])) |
    (~wheat_matba_march['date'].dt.month.isin([1, 2, 3]) & (wheat_matba_march['date'].dt.year +1 == wheat_matba_march['contract_year']))
]
wheat_matba_march = wheat_matba_march.sort_values(by='date')


march_matba = px.line(
    wheat_matba_march,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='March wheat MATBA contract',
)

march_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
march_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in march_matba.data:
    if trace.name == 'March2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'March2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

march_matba.show()


march_matba_html = march_matba.to_html(include_plotlyjs='cdn', full_html=True)



### JULY

In [0]:
wheat_matba_july=wheat_matba[wheat_matba['contract_month']==7]

wheat_matba_july['Season'] = 'July'+ wheat_matba_july['contract_year'].astype(str)
  
def adjust_virtual_date_wheat_july(row):
    if row['virtual_date'].month in [1, 2,3,4,5,6,7]:
        # Subtract 1 year from the year if month is November or julyember
        return row['virtual_date'] + pd.DateOffset(years=1)
    else:
        # Keep the date as is if the month is not November or julyember
        return row['virtual_date'].replace(year=row['virtual_date'].year)
      
wheat_matba_july['virtual_date'] = wheat_matba_july.apply(adjust_virtual_date_wheat_july, axis=1)
wheat_matba_july = wheat_matba_july[wheat_matba_july['value'] != 0]

wheat_matba_july = wheat_matba_july[
    (wheat_matba_july['date'].dt.month.isin([1, 2, 3,4,5,6,7]) & (wheat_matba_july['date'].dt.year == wheat_matba_july['contract_year'])) |
    (~wheat_matba_july['date'].dt.month.isin([1, 2, 3,4,5,6,7]) & (wheat_matba_july['date'].dt.year +1 == wheat_matba_july['contract_year']))
]
wheat_matba_july = wheat_matba_july.sort_values(by='date')


july_matba = px.line(
    wheat_matba_july,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'Spot Price',
        'season': 'Year'
    },
    title='July wheat MATBA contract',
)

july_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
july_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in july_matba.data:
    if trace.name == 'July2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'July2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

july_matba.show()


july_matba_html = july_matba.to_html(include_plotlyjs='cdn', full_html=True)



### DEC

In [0]:
wheat_matba_dec=wheat_matba[wheat_matba['contract_month']==12]

wheat_matba_dec['Season'] = 'Dec'+ wheat_matba_dec['contract_year'].astype(str)
wheat_matba_dec = wheat_matba_dec[wheat_matba_dec['date'].dt.year == wheat_matba_dec['contract_year']]
wheat_matba_dec = wheat_matba_dec[wheat_matba_dec['value'] != 0]

wheat_matba_dec = wheat_matba_dec.sort_values(by='date')


dec_matba = px.line(
    wheat_matba_dec,
    x='virtual_date',
    y='value',
    color='Season',
    labels={
        'virtual_date': '',
        'value': 'December contract',
        'season': 'Year'
    },
    title='December MATBA contract',
)

dec_matba.update_traces(hovertemplate='%{x|%d-%b}<br>%{y:.2f}<extra>%{fullData.name}</extra>')
dec_matba.update_layout(
    xaxis=dict(
        tickformat='%b',  # Month format like Jan, Feb...
        dtick="M1",
        hoverformat='%d-%b'
    ),
    yaxis_title='Price',
    template='plotly_white',
    height=700,
    width=1800,
    hovermode='x unified'
    
)

# Customize specific seasons
for trace in dec_matba.data:
    if trace.name == 'Dec2026':
        trace.line.width = 4
        trace.line.color = 'red'
        trace.line.dash = 'dash'
    elif trace.name == 'Dec2025':
        trace.line.width = 4
        trace.line.color = 'black'
        trace.line.dash = 'solid'

dec_matba.show()


dec_matba_html = dec_matba.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
wheat_mkt_spnapshot  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Wheat Market Snapshot Report</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color:red; /* change to any HEX or named color */
            text-decoration: underline; /* adds underline */
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Wheat Market Snapshot Report</h1>
    <h2>KC Wheat</h2>
        <h3>HRW spot price</h3>
            <div class="chart-container">{kc_hrw_spot_html}</div>
        <h3>HRW price evolution</h3>
            <div class="chart-container">{hrw_contract_ev_html}</div>
        <h3>HRW Calendar Spreads</h3>
            <div class="chart-container">{kc_cal_spread_html}</div>

    <h2>CBOT SRW Wheat</h2>
        <h3>CBOT SRW spot price</h3>
            <div class="chart-container">{cbot_srw_spot_html}</div>
        <h3>CBOT SRW price evolution</h3>
            <div class="chart-container">{srw_cbot_contract_ev_html}</div>
        <h3>CBOT SRW Calendar Spreads</h3>
            <div class="chart-container">{cbot_cal_spread_html}</div>


    <h2>SPREADS KC VS CBOT</h2>
        <div class="chart-container">{wheat_spreads_contract_ev_html}</div>

    <h2>CBOT WHEAT VS CBOT CORN</h2>
        <div class="chart-container">{cbot_spreads_contract_ev_html}</div>
        
    <h2>MATIF</h2>
        <h3>MATIF spot price</h3>
            <div class="chart-container">{matif_usd_spot_html}</div>
        <h3>MATIF price evolution</h3>
            <div class="chart-container">{matif_contract_ev_html}</div>
        <h3>MATIF Calendar Spreads</h3>
            <div class="chart-container">{matif_cal_spread_html}</div>
    <h2>EURUSD</h2>
        <h3>EURSUD spot price</h3>
            <div class="chart-container">{eurusd_spot_html}</div>   
    <h2>FOB SPOT COMPARISON</h2>
        <h3>Origins: UPR 11.5, Texas Gulf 11, Nikolaev 11.5, Novo 12.5, Kwinana 11 </h3>
            <div class="chart-container">{spot_chart_html}</div> 
    <h2>MATBA FLAT PRICES</h2>
        <h3>March</h3>
            <div class="chart-container">{march_matba_html}</div>
        <h3>July</h3>
            <div class="chart-container">{july_matba_html}</div> 
        <h3>Dec</h3>
            <div class="chart-container">{dec_matba_html}</div> 
</body>
</html>
"""
wheat_mkt_spnapshot_report_bytes = wheat_mkt_spnapshot.encode("utf-8")

grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=grains,
    subject=f'Wheat Market Snapshot {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body='Please find the report attached',
    attachment={"wheat_market_snapshot.html": wheat_mkt_spnapshot_report_bytes}
)
